# De TF-IDF a una clasificación aprendida

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Pregunta

¿Cómo aprende un clasificador a relacionar textos y etiquetas?

## Objetivo

Entrenar un clasificador con textos etiquetados, leer qué asoció a cada etiqueta y comprobar hasta dónde se puede confiar en lo que aprendió.

## Al terminar este cuaderno vas a poder:

1. Identificar en un corpus las observaciones, las características y las etiquetas.
2. Explicar por qué se separa un conjunto de entrenamiento de uno de prueba.
3. Leer una representación TF-IDF y los coeficientes que aprendió un clasificador, sin confundirlos con significados universales.
4. Comparar un modelo con palabras sueltas contra uno con palabras sueltas y pares de palabras, usando evidencia.

## Alcance

El corpus es pequeño, construido y repetitivo. Entrenaremos un clasificador para comprender el mecanismo, no para estimar rendimiento sobre cualquier corpus en español.

## ◈ Marco contextual

**Analogía.** En el cuaderno 01 escribimos a mano la planilla de puntos de cada palabra. Ahora imaginá que le damos a alguien 24 oraciones ya clasificadas por una persona y le pedimos que arme sola la planilla: qué palabras suelen aparecer en las oraciones positivas, en las negativas y en las neutras. Después le damos oraciones nuevas y le pedimos que aplique su planilla. Eso es *aprender de ejemplos*: la planilla la arma el programa; las etiquetas, las pusimos nosotros.

**Bajada tecnológica.** Los filtros de correo no deseado, la clasificación automática de tickets de soporte y la asignación de temas a noticias funcionan con este mismo esquema: texto convertido en números, más un clasificador que ajusta pesos. TF-IDF y la regresión logística son piezas clásicas que todavía se usan como punto de partida (*baseline*) antes de probar modelos más complejos.

## ◈ Microglosario

| Término | Qué significa en este cuaderno |
|---|---|
| **Observación** | Cada ejemplo del corpus: un texto. |
| **Característica** | Cada número que describe a una observación; acá, el peso de una palabra en ese texto. |
| **Etiqueta** | La respuesta correcta según el protocolo: `POS`, `NEG` o `NEU`. |
| **Entrenamiento y prueba** | Entrenamiento: textos con etiqueta que el modelo ve para ajustarse. Prueba: textos que reservamos para evaluar sin que el modelo los haya visto. |
| **TF-IDF** | Peso que sube si una palabra aparece mucho en un texto y baja si aparece en casi todos los textos del corpus. |
| **Coeficiente** | Número que el clasificador ajusta para cada combinación de palabra y clase. Es la "planilla" que arma el programa. |

## 1. Configuración del entorno

**Operación.** Importar las herramientas y registrar las versiones.

**Por qué importa.** Los resultados dependen de las versiones de las librerías. Este cuaderno no usa números aleatorios, así que con los mismos datos y versiones siempre vas a obtener el mismo resultado.

In [ ]:
# sys e importlib.metadata: para registrar qué versiones estamos usando.
import sys
import importlib.metadata

# pandas: tablas. display: las muestra completas en el cuaderno.
import pandas as pd
from IPython.display import display

# scikit-learn: TfidfVectorizer pasa textos a números; LogisticRegression es el clasificador;
# classification_report calcula las medidas de evaluación.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# utils.py (en esta carpeta): ayudas que dibujan los gráficos. No calculan nada.
from utils import (
    ETIQUETAS, COLOR_GENERAL,
    dibujar_barras, dibujar_coeficientes, dibujar_matriz_de_confusión, dibujar_probabilidades,
)

pd.set_option("display.max_colwidth", 140)

print("Python:", sys.version.split()[0])
for paquete in ["pandas", "scikit-learn", "matplotlib"]:
    print(f"{paquete}: {importlib.metadata.version(paquete)}")

**Leé el resultado.** Tienen que aparecer las tres versiones, sin errores.

## 2. El corpus y la separación entre entrenamiento y prueba

**Ficha de los datos.** Es el mismo corpus del cuaderno 01 (ficha completa ahí): 33 oraciones construidas por el docente para esta actividad, no testimonios reales. Las etiquetas son propuestas docentes que requieren discusión.

**Decisión.** La partición de prueba es deliberadamente fácil y comparte vocabulario con el entrenamiento. Permite comparar mecanismos, pero no estimar rendimiento general sobre textos argentinos. Los desafíos se mantienen aparte, sin etiquetas de referencia automáticas.

In [ ]:
corpus = pd.read_csv("datos/corpus_sentimiento.csv")
desafíos = pd.read_csv("datos/desafios.csv")

# loc con una condición se queda solo con las filas donde se cumple.
# copy() crea una copia, para no modificar el corpus original.
entrenamiento = corpus.loc[corpus["partición"] == "entrenamiento"].copy()
prueba = corpus.loc[corpus["partición"] == "prueba"].copy()

textos_de_entrenamiento = entrenamiento["texto"]
etiquetas_de_entrenamiento = entrenamiento["etiqueta_referencia"]
textos_de_prueba = prueba["texto"]
etiquetas_de_prueba = prueba["etiqueta_referencia"]

**Verificación.** Antes de seguir, confirmamos que los datos cumplen lo que esperamos.

In [ ]:
assert set(entrenamiento["id"]).isdisjoint(set(prueba["id"])), "Ningún texto puede estar a la vez en entrenamiento y en prueba."
assert corpus["etiqueta_referencia"].isin(ETIQUETAS).all(), "Las etiquetas solo pueden ser NEG, NEU o POS."

print(f"Textos de entrenamiento: {len(entrenamiento)}")
print(f"Textos de prueba: {len(prueba)}")
display(entrenamiento.head())

**Leé el resultado.** 24 textos para que el modelo aprenda y 9 reservados para evaluarlo. La separación importa porque un modelo siempre se ve bien con los textos que ya conoce: lo que interesa es qué hace con textos que no vio.

### ✎ Antes de entrenar

Elegí un ejemplo de cada clase del entrenamiento.

- Citá la expresión que justifica la etiqueta.
- Identificá el objeto valorado.
- Indicá si aceptarías la etiqueta de referencia.
- Proponé una situación en la que el contexto podría cambiar la interpretación.

Si cambiamos etiquetas, debemos registrar el criterio y aplicarlo consistentemente antes de evaluar.

**Mis anotaciones**

| Clase | Ejemplo elegido | Expresión que justifica | ¿Acepto la etiqueta? |
|---|---|---|---|
| `POS` | | | |
| `NEG` | | | |
| `NEU` | | | |

## 3. De texto a números: TF-IDF

**Pregunta.** ¿Cómo se convierte un texto en una fila de números que un clasificador pueda usar?

**La idea.** TF-IDF le da a cada palabra un peso en cada texto, combinando dos cosas:

- **TF** (frecuencia en el texto): cuánto aparece la palabra en *ese* texto.
- **IDF** (rareza en el corpus): cuán pocos textos del corpus la usan. Una palabra que está en casi todos (`la`) distingue poco; una que está en uno solo (`excelente`) distingue mucho.

**Parámetros de `TfidfVectorizer`**

| Parámetro | Qué significa | Para qué lo usamos | Qué cambia si lo modificás |
|---|---|---|---|
| `lowercase=True` | Pasa todo a minúsculas antes de contar | Que `La` y `la` cuenten como la misma palabra | Con `False`, `La` y `la` serían palabras distintas |
| `ngram_range=(1, 1)` | Cuántas palabras seguidas forma cada característica: de 1 a 1 | Usar solo palabras sueltas | Con `(1, 2)` también cuenta pares de palabras seguidas, como `no fue` |

**Dos operaciones distintas, en dos líneas.** `fit` aprende el vocabulario y cuán rara es cada palabra, **solo con el entrenamiento**. `transform` convierte textos en números usando lo ya aprendido. A los textos de prueba solo les aplicamos `transform`: no volvemos a ajustar con ellos.

In [ ]:
vectorizador = TfidfVectorizer(lowercase=True, ngram_range=(1, 1))

# fit aprende el vocabulario y la rareza de cada palabra, usando SOLO el entrenamiento.
vectorizador.fit(textos_de_entrenamiento)

# transform convierte los textos en números usando lo que fit aprendió.
tfidf_de_entrenamiento = vectorizador.transform(textos_de_entrenamiento)
tfidf_de_prueba = vectorizador.transform(textos_de_prueba)

vocabulario = vectorizador.get_feature_names_out()

print(f"Entrenamiento: {tfidf_de_entrenamiento.shape[0]} textos (filas) x {tfidf_de_entrenamiento.shape[1]} palabras distintas (columnas)")
print(f"Prueba: {tfidf_de_prueba.shape[0]} textos (filas) x {tfidf_de_prueba.shape[1]} palabras distintas (columnas)")

### ✎ Para pensar

- ¿Qué representa una fila de `tfidf_de_entrenamiento`? ¿Y una columna?
- ¿Por qué ambas matrices tienen la misma cantidad de columnas?
- ¿Qué puede ocurrir con una palabra que nunca apareció en el entrenamiento?

Respuestas:

In [ ]:
posición = 0

# La matriz guarda solo los valores distintos de cero; toarray() la pasa a una lista de
# números común para poder verlos todos, y [0] toma la única fila que pedimos.
pesos_del_texto = tfidf_de_entrenamiento[posición].toarray()[0]

tabla_de_pesos = pd.DataFrame({"palabra": vocabulario, "peso": pesos_del_texto})

# Nos quedamos solo con las palabras que están en el texto (peso mayor que 0).
tabla_de_pesos = tabla_de_pesos.loc[tabla_de_pesos["peso"] > 0]

print("Texto:", entrenamiento.iloc[posición]["texto"])
dibujar_barras(
    tabla=tabla_de_pesos, columna_nombre="palabra", columna_valor="peso",
    título="Peso TF-IDF de cada palabra en este texto", color=COLOR_GENERAL,
)

**Leé el gráfico.** La palabra que más pesa es `excelente`, después `atención`, y recién después `fue` y `la`.

- **Observación:** las palabras que aparecen en pocos textos del corpus pesan más que las que aparecen en muchos.
- **Interpretación posible:** `excelente` identifica a este texto casi sin ayuda; `la` y `fue` se repiten en tantos textos que aportan poco para distinguirlos.
- **Límite:** que una palabra pese mucho en un texto no quiere decir que sea importante para la valoración; solo que es poco común en este corpus.

**Probá.** Cambiá `posición = 0` por otro número entre 0 y 23 y mirá qué palabras pesan más en otro texto.

## 4. Entrenar el clasificador

**Pregunta.** ¿Cómo se pasa de números por texto a una decisión `POS`, `NEG` o `NEU`?

**Cómo funciona, con la analogía de la planilla.** El clasificador (una *regresión logística*) arma, para cada clase, una planilla con un peso por palabra. Para clasificar un texto suma los pesos de las palabras que aparecen (multiplicados por su TF-IDF) y elige la clase con más puntos. La diferencia con el cuaderno 01 es que **esos pesos los ajusta el programa** para acertar en los ejemplos de entrenamiento: nadie escribió regla alguna para cada palabra.

**Parámetro.** `max_iter=1000` es cuántas rondas de ajuste puede hacer como máximo. Si fuera demasiado bajo, el programa avisaría que no alcanzó a terminar.

In [ ]:
clasificador = LogisticRegression(max_iter=1000)

# fit ajusta los coeficientes: busca los valores que mejor reproducen las etiquetas del entrenamiento.
clasificador.fit(X=tfidf_de_entrenamiento, y=etiquetas_de_entrenamiento)

print("Clases:", clasificador.classes_)
print("Coeficientes (clases x palabras):", clasificador.coef_.shape)

**Leé el resultado.** El clasificador aprendió 3 filas de coeficientes (una por clase) con 60 valores cada una (uno por palabra del vocabulario).

El algoritmo aprende de las etiquetas suministradas. No valida por sí mismo nuestro protocolo de anotación.

### ✎ Antes de predecir

Leé tres textos de prueba y anticipá:

- Qué etiqueta deberían recibir según el protocolo.
- Qué palabras del entrenamiento podrían ayudar al modelo.
- Qué aspecto podría dificultar la clasificación.

Predicciones:

## 5. Evaluar con la partición de prueba

**Operación.** Pedirle al modelo una etiqueta para cada texto de prueba y compararla con la etiqueta de referencia.

In [ ]:
# predict devuelve la etiqueta con más puntos para cada texto.
predicciones = clasificador.predict(tfidf_de_prueba)

resultados = prueba.copy()
resultados["predicción"] = predicciones
resultados["acierto"] = resultados["etiqueta_referencia"] == resultados["predicción"]

display(resultados[["id", "texto", "etiqueta_referencia", "predicción", "acierto"]])

cantidad_de_aciertos = resultados["acierto"].sum()
print(f"Aciertos: {cantidad_de_aciertos} de {len(resultados)}")

**Los mismos resultados, en una matriz.** Las filas son la referencia y las columnas, la respuesta del modelo.

In [ ]:
dibujar_matriz_de_confusión(
    referencia=etiquetas_de_prueba, predicción=predicciones,
    título="Prueba: clasificador con palabras sueltas",
)

**Leé la matriz.** Los aciertos están en la diagonal. Los 9 textos de prueba se clasificaron bien.

### Cómo leer las medidas

- **Precisión:** entre los textos que el modelo puso en una clase, qué proporción pertenece realmente a ella.
- **Exhaustividad:** entre los textos que pertenecen a una clase, qué proporción encontró el modelo.
- **F1:** un único número que combina precisión y exhaustividad.
- **Cantidad:** cuántos textos de referencia hay de esa clase.

In [ ]:
informe = classification_report(
    y_true=etiquetas_de_prueba, y_pred=predicciones,
    labels=ETIQUETAS, output_dict=True, zero_division=0,
)

# El informe viene como diccionario; lo pasamos a tabla y nos quedamos con las tres clases.
tabla_del_informe = pd.DataFrame(informe).T.loc[ETIQUETAS]
tabla_del_informe.columns = ["precisión", "exhaustividad", "F1", "cantidad"]
tabla_del_informe["cantidad"] = tabla_del_informe["cantidad"].astype(int)
display(tabla_del_informe.round(2))

- **Observación:** precisión, exhaustividad y F1 valen 1,00 en las tres clases.
- **Interpretación posible:** la prueba es fácil y comparte vocabulario con el entrenamiento; el modelo se apoyó en palabras que ya conocía.
- **Límite:** con tres textos por clase, un solo error cambia mucho las medidas. Un 1,00 en esta prueba no garantiza buen desempeño con textos reales.

### ✎ Para pensar

Elegí un casillero de la matriz y explicalo con un texto concreto. Si no hubo errores, explicá por qué una prueba pequeña y sencilla no garantiza que el modelo generalice.

Respuesta:

## 6. Qué asoció el modelo con cada clase

**Pregunta.** ¿Qué palabras empujan hacia cada etiqueta?

**Operación.** Armamos una tabla de coeficientes (clases x palabras) y miramos, para cada clase, las 8 palabras con coeficiente más alto. Un coeficiente alto significa que, cuando esa palabra está presente, el modelo suma puntos a esa clase.

In [ ]:
# Una tabla con una fila por clase y una columna por palabra del vocabulario.
coeficientes_por_clase = pd.DataFrame(
    clasificador.coef_, index=clasificador.classes_, columns=vocabulario
)

tablas_por_clase = {}
for clase in ETIQUETAS:
    coeficientes_de_la_clase = coeficientes_por_clase.loc[clase]
    más_altos = coeficientes_de_la_clase.sort_values(ascending=False).head(8)
    tablas_por_clase[clase] = pd.DataFrame(
        {"característica": más_altos.index, "coeficiente": más_altos.values}
    )

dibujar_coeficientes(tablas_por_clase=tablas_por_clase)

**Leé los tres gráficos.**

- **Observación:** en `NEG` y `POS` las palabras más asociadas son valorativas (`mala`, `inútil`, `pésima` / `buena`, `clara`, `excelente`). En `NEU` aparecen palabras temáticas o administrativas (`tiene`, `tres`, `secciones`, `reunión`, `correo`).
- **Observación 2:** `fue` figura entre las ocho primeras de `NEG` y de `POS`, con el mismo coeficiente (0,34), y en `NEU` su coeficiente es negativo.
- **Interpretación posible:** `fue` no valora nada. El modelo la asocia con `NEG` y `POS` porque, en este corpus, aparece en textos valorativos ("La atención *fue* pésima") y nunca en los neutros. Es un rasgo del corpus, no del idioma.
- **Límite:** un coeficiente alto describe una asociación en estos 24 textos. No convierte a la palabra en positiva o negativa en todos sus usos, ni expresa una relación causal.

### ✎ Para pensar

1. ¿Qué ejemplo nuevo podría poner a prueba la asociación entre `fue` y las clases valorativas?
2. ¿Cómo pudo producir el corpus las palabras temáticas de `NEU`?

## 7. Más contexto: palabras sueltas y pares de palabras

**Hipótesis.** "Agregar pares de palabras (bigramas) permitirá distinguir afirmaciones positivas y sus negaciones".

Antes de probarla, la hipótesis necesita condiciones. Revisemos el entrenamiento: ¿contiene negaciones?

In [ ]:
# str.contains busca un fragmento en cada texto; \bno\b es la palabra "no" completa.
hay_negación = entrenamiento["texto"].str.contains(pat=r"\bno\b", case=False)
cantidad_con_negación = hay_negación.sum()
print(f"Textos de entrenamiento que contienen 'no': {cantidad_con_negación} de {len(entrenamiento)}")

**Leé el resultado.** Ningún texto de entrenamiento contiene la palabra `no`. Aunque la representación con bigramas *pueda* conservar la secuencia `no fue`, el modelo no tiene ejemplos de los que aprender qué hacer con ella.

Una representación capaz de conservar una secuencia no garantiza que existan ejemplos suficientes para aprender su función.

**Operación.** Igual entrenamos un segundo modelo con `ngram_range=(1, 2)` para comparar.

In [ ]:
vectorizador_con_bigramas = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
vectorizador_con_bigramas.fit(textos_de_entrenamiento)
tfidf_de_entrenamiento_con_bigramas = vectorizador_con_bigramas.transform(textos_de_entrenamiento)
tfidf_de_prueba_con_bigramas = vectorizador_con_bigramas.transform(textos_de_prueba)

clasificador_con_bigramas = LogisticRegression(max_iter=1000)
clasificador_con_bigramas.fit(X=tfidf_de_entrenamiento_con_bigramas, y=etiquetas_de_entrenamiento)
predicciones_con_bigramas = clasificador_con_bigramas.predict(tfidf_de_prueba_con_bigramas)

print("Características con palabras sueltas:", tfidf_de_entrenamiento.shape[1])
print("Características con palabras sueltas y pares:", tfidf_de_entrenamiento_con_bigramas.shape[1])

**Comparación.** Ponemos lado a lado las respuestas de los dos modelos.

In [ ]:
comparación = prueba[["id", "texto", "etiqueta_referencia"]].copy()
comparación["palabras sueltas"] = predicciones
comparación["con pares"] = predicciones_con_bigramas

aciertos_sueltas = (comparación["etiqueta_referencia"] == comparación["palabras sueltas"]).sum()
aciertos_con_pares = (comparación["etiqueta_referencia"] == comparación["con pares"]).sum()
print(f"Aciertos con palabras sueltas: {aciertos_sueltas} de {len(comparación)}")
print(f"Aciertos con pares de palabras: {aciertos_con_pares} de {len(comparación)}")
display(comparación)

**Leé la tabla.**

- **Observación:** con pares de palabras el modelo pasó de 9 a 7 aciertos. Dos textos (`P02`, `P05`) que antes acertaba, ahora los clasifica como `NEU`. Las características pasaron de 60 a 123.
- **Interpretación posible (hipótesis, sin probar):** casi todos los pares aparecen en un solo texto del entrenamiento, así que aportan poco para generalizar y diluyen el peso de las palabras sueltas que sí son informativas.
- **Límite:** con 9 textos de prueba, que 2 cambien ya es mucho en términos de porcentaje y poco en términos de evidencia. No alcanza para decir que los bigramas "empeoran" el método en general.

La hipótesis inicial no se cumplió: no solo no resolvió la negación, sino que además empeoró casos que ya andaban.

## 8. Los desafíos y una palabra que el modelo no conoce

**Operación.** Aplicamos ambos modelos a los 8 desafíos y miramos, además de la etiqueta, **cuánta seguridad** tiene el modelo: `predict_proba` devuelve cómo reparte el 100 % entre las tres clases.

In [ ]:
tfidf_de_desafíos = vectorizador.transform(desafíos["texto"])
tfidf_de_desafíos_con_bigramas = vectorizador_con_bigramas.transform(desafíos["texto"])

revisión = desafíos.copy()
revisión["palabras sueltas"] = clasificador.predict(tfidf_de_desafíos)
revisión["con pares"] = clasificador_con_bigramas.predict(tfidf_de_desafíos_con_bigramas)
display(revisión)

**Seguridad del modelo.** Además de la etiqueta, miramos cómo reparte su 100 % entre las clases.

In [ ]:
# predict_proba reparte el 100 % entre las clases, en el orden de clasificador.classes_.
assert list(clasificador.classes_) == ETIQUETAS, "El orden de las clases cambió."
probabilidades = clasificador.predict_proba(tfidf_de_desafíos)

tabla_de_probabilidades = pd.DataFrame(
    probabilidades, columns=["prob_NEG", "prob_NEU", "prob_POS"]
)
tabla_de_probabilidades["texto"] = desafíos["texto"]

dibujar_probabilidades(
    tabla=tabla_de_probabilidades, columna_texto="texto",
    título="Cómo reparte el modelo (palabras sueltas) su 100 % en cada desafío",
)

**Leé la tabla y el gráfico.**

- **Observación:** "La atención no fue buena" (D01) recibe `POS`. Varios desafíos tienen barras casi parejas: D04, D05, D07 y D08 reparten el 100 % entre dos o tres clases con porcentajes muy cercanos.
- **Interpretación posible:** el modelo no ve la negación (no tuvo ejemplos con `no`), y cuando las probabilidades están tan juntas la etiqueta que devuelve `predict` es casi un desempate, no una decisión firme.
- **Límite:** una etiqueta sola esconde esa incertidumbre. Dos modelos que dan la misma etiqueta pueden tener niveles de seguridad muy distintos.

### Un texto con vocabulario desconocido

¿Qué hace el modelo con "La interfaz es intuitiva y accesible"? Es claramente favorable, pero `interfaz`, `intuitiva` y `accesible` no están en el entrenamiento.

In [ ]:
texto_nuevo = "La interfaz es intuitiva y accesible."

# build_analyzer devuelve la función que corta el texto igual que lo hace el vectorizador.
cortar_en_tokens = vectorizador.build_analyzer()
tokens_del_texto = cortar_en_tokens(texto_nuevo)

conocidas = []
desconocidas = []
for token in tokens_del_texto:
    if token in vocabulario:
        conocidas.append(token)
    else:
        desconocidas.append(token)

print("Palabras que el modelo conoce:", conocidas)
print("Palabras que el modelo no conoce:", desconocidas)

tfidf_del_texto_nuevo = vectorizador.transform([texto_nuevo])
probabilidades_del_texto = clasificador.predict_proba(tfidf_del_texto_nuevo)[0]
print("Predicción:", clasificador.predict(tfidf_del_texto_nuevo)[0])

tabla_del_texto = pd.DataFrame({"clase": clasificador.classes_, "probabilidad": probabilidades_del_texto})
display(tabla_del_texto.round(2))

**Leé el resultado.**

- **Observación:** el modelo solo reconoce `la` y `es`. Las tres palabras que expresan la valoración (`interfaz`, `intuitiva`, `accesible`) son invisibles para él. `NEG` y `POS` quedan empatadas en 37 % y `NEU` en 27 %; la etiqueta `NEG` sale de un desempate.
- **Interpretación posible:** cuando un texto no contiene palabras conocidas con peso, el modelo no tiene en qué apoyarse y decide con lo que tiene: palabras vacías.
- **Límite:** esto no dice que el modelo "piense" que la interfaz es mala. Dice que no tiene información para opinar y que `predict` igual devuelve una etiqueta.

### ✎ Para pensar

1. ¿Qué habría que agregar al entrenamiento para que el modelo reconozca este texto como favorable?
2. ¿Qué consecuencias tendría usar un modelo así con opiniones reales sin mirar nunca las probabilidades?

## 9. Experimento propio

**Producto.** Un conjunto de ejemplos nuevos con negaciones, un modelo reentrenado y una comparación antes y después.

Primero escribí tu criterio de etiquetado y qué resultado esperás. Después completá la tabla de ejemplos nuevos de la celda siguiente.

### ✎ Antes de ejecutar

1. ¿Con qué criterio etiquetás una oración con negación (por ejemplo, "no fue clara")?
2. ¿Qué resultado esperás en D01 y D02 después de reentrenar?
3. ¿En qué caso la ampliación podría empeorar el resultado?

**Reglas para no hacer trampa.**

- Los ejemplos nuevos son *desarrollo*: sirven para ajustar el modelo.
- **No uses** las oraciones de los desafíos ni variantes casi idénticas (cambiar una sola palabra). Si lo hacés, ya no estás evaluando: estás copiando la respuesta.
- No uses la prueba repetidamente para elegir el modelo.

In [ ]:
# COMPLETÁ: agregá tus ejemplos nuevos. Cada fila es (identificador, texto, etiqueta).
# Las dos primeras son de muestra; podés reemplazarlas o agregar más.
ampliación = pd.DataFrame([
    ("N01", "La explicación no fue clara.", "NEG"),
    ("N02", "El trámite no resultó complicado.", "POS"),
], columns=["id", "texto", "etiqueta_referencia"])

assert ampliación["etiqueta_referencia"].isin(ETIQUETAS).all(), "Las etiquetas solo pueden ser NEG, NEU o POS."
display(ampliación)

**Reentrenar.** Con tus ejemplos agregados, entrenamos de nuevo.

In [ ]:
# Juntamos el entrenamiento original con los ejemplos nuevos y entrenamos de nuevo, con pares de palabras.
entrenamiento_ampliado = pd.concat(
    [entrenamiento[["id", "texto", "etiqueta_referencia"]], ampliación], ignore_index=True
)

vectorizador_ampliado = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
tfidf_ampliado = vectorizador_ampliado.fit_transform(entrenamiento_ampliado["texto"])
clasificador_ampliado = LogisticRegression(max_iter=1000)
clasificador_ampliado.fit(X=tfidf_ampliado, y=entrenamiento_ampliado["etiqueta_referencia"])

**Operación.** Comparamos el modelo de antes con el ampliado en los dos desafíos de negación y en la prueba.

In [ ]:
negaciones = desafíos.loc[desafíos["problema"] == "Negación"].copy()
negaciones["antes (palabras sueltas)"] = clasificador.predict(vectorizador.transform(negaciones["texto"]))
negaciones["después (ampliado)"] = clasificador_ampliado.predict(vectorizador_ampliado.transform(negaciones["texto"]))
display(negaciones[["id", "texto", "antes (palabras sueltas)", "después (ampliado)"]])

predicciones_ampliado = clasificador_ampliado.predict(vectorizador_ampliado.transform(textos_de_prueba))
aciertos_ampliado = (etiquetas_de_prueba == predicciones_ampliado).sum()
print(f"Aciertos en la prueba con el modelo ampliado: {aciertos_ampliado} de {len(prueba)}")

**Leé el resultado.**

- ¿Cambió D01 o D02? ¿Cambió en la dirección que esperabas?
- ¿Cuántos ejemplos nuevos necesitaste agregar antes de ver algún efecto?
- ¿Empeoró algo en la prueba?

Escribí qué resultado esperabas y qué obtuviste, aunque no coincidan.

**Con los dos ejemplos de muestra** (sin agregar nada): D01 y D02 no cambian y la prueba baja de 9 a 8 aciertos. Dos oraciones no alcanzan para enseñar una construcción, y sumar ejemplos puede alterar lo que ya funcionaba. Probá con más ejemplos propios, distintos entre sí, y fijate cuántos hacen falta.

## Cierre

### Qué aprendimos

| Idea | Evidencia en este cuaderno |
|---|---|
| El modelo aprende de las etiquetas que le damos. | Los coeficientes reflejan el corpus (`fue`, palabras temáticas en `NEU`). |
| Representar una secuencia no es aprender su función. | Los bigramas no resolvieron la negación: el entrenamiento no tenía ningún `no`. |
| Una etiqueta esconde la incertidumbre. | En varios desafíos las probabilidades quedaron casi empatadas. |
| Una prueba fácil no mide generalización. | 9 de 9 con palabras sueltas, y errores en los desafíos. |

### Limitaciones del método

- Corpus pequeño, construido y repetitivo.
- Palabras que no estaban en el entrenamiento son invisibles.
- El orden de las palabras solo se captura con pares, y solo si hay ejemplos.

### Conclusión: completá con tus palabras

- El clasificador aprendió a partir de...
- TF-IDF representó...
- Una asociación que pude explicar fue...
- Un resultado que contradijo mi expectativa fue...
- La comparación de bigramas mostró...
- Para estudiar textos auténticos todavía necesitaríamos...

### Conexión con el próximo cuaderno

Este modelo aprendió de 24 oraciones. En el cuaderno **03** vamos a usar un modelo preentrenado con muchísimos más textos, y en lugar de entrenarlo, lo vamos a **auditar**: qué hace bien, qué hace mal y qué no podemos saber de él.